# Session 3 — Signal Region and Analysis Strategy

Here we combine the object selections from Session 2 into a **full signal region (SR)** for the bb + MET search. We will also look at how the main backgrounds populate the SR and how control regions are used conceptually.

**Learning objectives:**
- Define and implement the bb + MET signal region for the bbDM search
- Apply full event selection and study the resulting MET distribution
- Compare event yields before and after cuts (cutflow and yield tables)
- Understand background composition and the role of control regions

<div style="display: flex; gap: 0; align-items: flex-start; justify-content: center; margin-top: 0.75rem; flex-wrap: wrap;">
  <div style="flex: 1 1 45%; min-width: 260px; text-align: right;">
    <img src="figures/session3_fig_regions_cartoon.png" alt="Regions cartoon (SR and control regions)" style="width: 100%; max-width: 480px; height: auto; display: block; margin: 0 auto;"/>
  </div>
  <div style="flex: 1 1 45%; min-width: 260px; text-align: left;">
    <img src="figures/session3_fig_sr_cartoon.png" alt="Signal region cartoon" style="width: 100%; max-width: 480px; height: auto; display: block; margin: 0 auto;"/>
  </div>
</div>


---
## 1. Signal Region Definition

We define the **signal region (SR)** as:

- **Trigger:** pass at least one of the analysis HLT paths (MET / muon )

- **2 ≤ Njets ≤ 3**, with the two leading jets b-tagged (medium WP)

- **0 isolated leptons** (lepton veto)

- **Δφ(jet, MET) > 0.5**

- **$\Delta p_\mathrm{T}^\mathrm{miss}(\mathrm{PF} - \mathrm{calo}) < 0.5$ GeV** (after Δφ; before the MET cut; skipped if CaloMET is absent from the file)

- **MET > 250 GeV**

This targets the signature $pp → b\bar{b}+\chi\bar{\chi}$ while suppressing many backgrounds.

---
## 2. Backgrounds in the Signal Region

| Background | Why it enters | How we suppress |
|------------|----------------|------------------|
| **$t\bar{t}$** | Has b-jets; semileptonic has MET from ν | Lepton veto removes most; MET cut reduces rest |
| **Z $(→\nu\bar{\nu})$ +jets** | Invisible $\nu\bar{\nu}→$ large MET | Dominant at high MET; need MC/data-driven estimate |
| **W + jets** | W $→$ $\ell$ $\nu$ gives MET | Lepton veto removes W $→$ $\ell$ $\nu$ |


In [ ]:
# Ensure project root is on sys.path (for SWAN / any kernel)
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt

from config.datasets_2017 import get_short_datasets_meta, get_trigger_list
from processor.bbdm_processor import get_nanoevents, select_good_jets, count_bjets, n_tight_leptons, RECOIL_MIN, cos_theta_star, met_pf_calo_consistency_mask

# Normalization
LUMI_PB = 41.5 * 1000

# Load one file per dataset from config/datasets_2017_short.yaml (limit to 10k events per file)
MAX_EVENTS = 10_000
datasets = get_short_datasets_meta()

events_by_dataset = {}
labels = {}
xsecs = {}
is_data = {}
norm_factors = {}

for name, meta in datasets.items():
    labels[name] = meta.get("label", name)
    xsecs[name] = meta.get("xsec")
    is_data[name] = bool(meta.get("isData", False))
    path_list = meta.get("files", [])
    if path_list:
        ev = get_nanoevents(path_list[0])[:MAX_EVENTS]
        events_by_dataset[name] = ev
        if not is_data[name]:
            xsec = xsecs.get(name)
            if xsec is None:
                norm_factors[name] = None
            else:
                norm_factors[name] = (float(xsec) * LUMI_PB) / float(len(ev))

# MC-only for SR: background names (exclude data and signal)
bkg_names = [k for k in events_by_dataset if (not is_data.get(k, False)) and ("bbDM" not in k)]

# Matplotlib legend labels (LaTeX)
latex_labels = {
    "DYJets": r"$Z(\ell\ell)+$jets",
    "ZJets": r"$Z(\nu\bar{\nu})+$jets",
    "WJets": r"$W(\ell\nu)+$jets",
    "DIBOSON": r"WW",
    "STop": r"Single $t$",
    "Top": r"$t\bar{t}$",
    "SMH": r"SMH",
    "bbDM_2HDMa_LO_5f": r"bbDM (2HDM+a)",
    "MET_Run2017B": r"Data",
}

# Sanity printout: xsec, N, normalization factor
for name in bkg_names:
    print(name, "xsec=", xsecs.get(name), "N=", len(events_by_dataset[name]), "norm=", norm_factors.get(name))

---
## 3. Full Event Selection

Combine all steps:
1. **Trigger** (OR of analysis HLT paths)

2. Good jets (pT, η, jetId); 2 ≤ Njets ≤ 3, leading two b-tagged

3. Count tight leptons (veto: require 0)

4. Δφ(jet, MET) > 0.5

5. $\Delta p_\mathrm{T}^\mathrm{miss}(\mathrm{PF} - \mathrm{calo}) < 0.5$

6. MET > 250 GeV

Implement this and count events passing each step (cutflow).

In [ ]:
# Run cutflow for each dataset (events_by_dataset and labels from setup cell)
def run_cutflow(events):
    # Trigger OR (first cut)
    trigger_list = get_trigger_list()
    hlt_fields = set(events.HLT.fields) if hasattr(events, "HLT") and hasattr(events.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(events.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | events.HLT[tname]
    n0 = int(ak.sum(trigger_mask))
    events = events[trigger_mask]

    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    nlep = n_tight_leptons(events)
    met = events.MET.pt
    met_phi = events.MET.phi
    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5

    jets2 = ak.pad_none(good_jets, 2)
    lead2_b = ak.fill_none(jets2[:, 0].btagDeepFlavB > 0.2783, False) & ak.fill_none(jets2[:, 1].btagDeepFlavB > 0.2783, False)
    jets_2to3 = (njets >= 2) & (njets <= 3)
    sr_jets = jets_2to3 & lead2_b

    n1 = ak.sum(sr_jets)
    n2 = ak.sum(sr_jets & (nlep == 0))
    n3 = ak.sum(sr_jets & (nlep == 0) & dphi_cut)
    met_pf_calo_ok = met_pf_calo_consistency_mask(events)
    n4 = ak.sum(sr_jets & (nlep == 0) & dphi_cut & met_pf_calo_ok)
    n5 = ak.sum(sr_jets & (nlep == 0) & dphi_cut & met_pf_calo_ok & (met > RECOIL_MIN))
    recoil_key = f"Recoil>{int(RECOIL_MIN)}"
    return {
        "trigger": n0,
        "presel": int(n1),
        "0lep": int(n2),
        "Δφ>0.5": int(n3),
        "|ΔMET(PF-calo)|<0.5": int(n4),
        recoil_key: int(n5),
    }

cutflow_by_dataset = {name: run_cutflow(ev) for name, ev in events_by_dataset.items()}
for name in list(cutflow_by_dataset.keys())[:3]:
    print(labels.get(name, name), cutflow_by_dataset[name])


### Exercise 3.1 — Cutflow
Implement the cutflow above and fill a table: **Cut** | **Yield**. Use your NanoAOD sample (one process is enough for the exercise).

In [ ]:
# Your code: cutflow table

---
## 4. MET Distribution in the Signal Region

**Note:** Signal-region plots are **MC-only**; no data is plotted in the signal region.

Plot **MET** for events that pass the full signal region (same cuts as the cutflow: after Δφ and PF–calo consistency, **MET > 250 GeV**, ≥2 b-jets, 0 leptons). Use bins starting at 200 GeV (e.g. 200–600 GeV in 50 GeV bins). Use `events_by_dataset` and `bkg_names` to stack background MC.

In [ ]:
# MET in SR: MC-only (use bkg_names); stack backgrounds (scaled to 41.5/fb)
bins_met = np.linspace(200, 600, 50)
met_arrays, w_arrays, leg_labels = [], [], []

trigger_list = get_trigger_list()
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue

    ev = events_by_dataset[name]
    hlt_fields = set(ev.HLT.fields) if hasattr(ev, "HLT") and hasattr(ev.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev.HLT[tname]
    ev = ev[trigger_mask]
    good_jets = select_good_jets(ev)
    njets = ak.num(good_jets)
    nlep = n_tight_leptons(ev)
    met = ev.MET.pt

    met_phi = ev.MET.phi
    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5

    jets2 = ak.pad_none(good_jets, 2)
    lead2_b = ak.fill_none(jets2[:, 0].btagDeepFlavB > 0.2783, False) & ak.fill_none(jets2[:, 1].btagDeepFlavB > 0.2783, False)

    sr_mask = (
        (njets >= 2)
        & (njets <= 3)
        & lead2_b
        & (nlep == 0)
        & dphi_cut
        & met_pf_calo_consistency_mask(ev)
        & (met > RECOIL_MIN)
    )
    recoil = met[sr_mask]
    flat = ak.to_numpy(ak.ravel(recoil))
    if len(flat) == 0:
        continue

    met_arrays.append(flat)
    w_arrays.append(np.full_like(flat, norm, dtype=float))
    leg_labels.append(latex_labels.get(name, labels.get(name, name)))

if met_arrays:
    plt.hist(met_arrays, bins=bins_met, weights=w_arrays, label=leg_labels, stacked=True, histtype="stepfilled", alpha=0.7)
plt.xlabel("MET [GeV]")
plt.ylabel("Events (scaled)")
plt.title("MET in signal region (MC only)")
plt.legend()
plt.show()

### Exercise 3.2 — MET after selection
Produce the MET distribution in the signal region. Add axis labels and a title. If you have multiple samples (e.g. tt̄, Z→νν), you can plot them stacked or overlaid.

In [ ]:
# Your code: MET in SR

---
## 5. cos θ* distribution in the signal region

The **cos θ*** observable is defined from the pseudorapidity difference of the two leading jets: **cos θ* = |tanh(Δη/2)|**, with Δη = η_jet0 − η_jet1. It is the main angular observable for the bb+MET search and is used in the fit and limit in Session 4.

Below we plot the cos θ* distribution for events passing the full signal region (same selection as the MET plot). Only MC backgrounds are stacked; the same luminosity scaling (41.5/fb) is applied.

In [ ]:
# cos(theta*) in SR (main observable): same SR mask as MET plot
trigger_list = get_trigger_list()
bins_cts = np.linspace(0, 1, 25)
cts_arrays, cts_w, cts_labels = [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    ev = events_by_dataset[name]
    hlt_fields = set(ev.HLT.fields) if hasattr(ev, "HLT") and hasattr(ev.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev.HLT[tname]
    ev = ev[trigger_mask]
    good_jets = select_good_jets(ev)
    njets = ak.num(good_jets)
    nlep = n_tight_leptons(ev)
    met = ev.MET.pt
    met_phi = ev.MET.phi
    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5
    jets2 = ak.pad_none(good_jets, 2)
    lead2_b = ak.fill_none(jets2[:, 0].btagDeepFlavB > 0.2783, False) & ak.fill_none(jets2[:, 1].btagDeepFlavB > 0.2783, False)
    sr_mask = (
        (njets >= 2)
        & (njets <= 3)
        & lead2_b
        & (nlep == 0)
        & dphi_cut
        & met_pf_calo_consistency_mask(ev)
        & (met > RECOIL_MIN)
    )
    good_jets_sr = good_jets[sr_mask]
    has_two = ak.num(good_jets_sr) >= 2
    jets_pad = ak.pad_none(good_jets_sr, 2)
    jet0 = jets_pad[:, 0]
    jet1 = jets_pad[:, 1]
    mask = has_two & ~ak.is_none(jet1)
    if ak.sum(mask) == 0:
        continue
    cts = cos_theta_star(jet0[mask], jet1[mask])
    flat = ak.to_numpy(ak.ravel(cts))
    if len(flat) == 0:
        continue
    cts_arrays.append(flat)
    cts_w.append(np.full_like(flat, norm, dtype=float))
    cts_labels.append(latex_labels.get(name, labels.get(name, name)))
if cts_arrays:
    plt.figure()
    plt.hist(cts_arrays, bins=bins_cts, weights=cts_w, label=cts_labels, stacked=True, histtype="stepfilled", alpha=0.7)
    plt.xlabel("$\cos(\theta)*$ (SR)")
    plt.ylabel("Events (scaled)")
    plt.title("$\cos(\theta)*$ in signal region (MC only)")
    plt.legend()
    plt.show()

---
## 6. Yields Before and After Cuts

Build a **yield table** with one row per cut and one column per process (if you have multiple samples). Example:

| Cut | $t\bar{t}$ | Z ( $→\nu\bar{\nu}$)+jets | W( $→\ell\nu$)+jets |
|-----|-----|------|--------|
| Trigger (OR) | ... | ... | ... |
| Preselection | ... | ... | ... |
| ≥2 b-jets | ... | ... | ... |
| 0 leptons | ... | ... | ... |
| Δφ > 0.5 | ... | ... | ... |
| ΔMET(PF−calo) < 0.5 | ... | ... | ... |
| MET > 250 GeV | ... | ... | ... |

In [ ]:
# Example for a single sample:
# import pandas as pd
# df = pd.DataFrame({
#     "Cut": ["Trigger", "Presel", "≥2 b-jets", "0 leptons", "Δφ>0.5", "|ΔMET(PF−calo)|<0.5", "MET>250"],
#     "Yield": [N0, N1, N2, N3, N4, N5, N6]  # keys: trigger, presel, 0lep, Δφ>0.5, |ΔMET(PF−calo)|<0.5, MET>250
# })
# print(df)

### Exercise 3.3 — Yield table
Create a yield table for your sample(s). Discuss: which background survives most in the SR? Why?

In [ ]:
# Your code: yield table

---
## 7. Control Regions

- **Goal:** use **control regions** to validate background modelling with data.
- **Regions in this notebook:** **Z (dilepton)** and **Top (single-lepton)**, following the reference analysis.

**Preselection:**

- Trigger ($p_\mathrm{T}^\mathrm{miss}$ or single electron).

- Jets: $\geq 1$, leading $p_\mathrm{T} > 100$ GeV, others $p_\mathrm{T} > 30$ GeV, $|\eta| < 2.5$.

- Overlap removal, photon veto.

- $\min \Delta\phi(\mathrm{jet}, p_\mathrm{T}^\mathrm{miss}) > 0.5$

- $\Delta p_\mathrm{T}^\mathrm{miss}(\mathrm{PF} - \mathrm{calo}) < 0.5$

- $p_\mathrm{T}^\mathrm{miss}$ or Recoil $> 250$ GeV.
    - Recoil: $\mathbf{U} = -(\vec{p}_\mathrm{T}^\mathrm{miss} + \sum \vec{p}_{\mathrm{T}}^{\,\ell})$.





**Z CR (dilepton):**

- On top of preselection: exactly two opposite-sign same-flavor leptons (ee or $\mu\mu$).

- Leading lepton $p_\mathrm{T} > 30$ GeV; $70 < m_{\ell\ell} < 110$ GeV.

- No b-tag requirement.

- **Data can be plotted here.**

**Top CR (single-lepton):**

- On top of preselection: exactly one lepton (e or $\mu$) with $p_\mathrm{T} > 30$ GeV.

- $m_\mathrm{T} < 160$ GeV with $m_\mathrm{T} = \sqrt{2\,p_\mathrm{T}^\mathrm{miss}\,p_{\mathrm{T},\ell}\big(1-\cos\Delta\phi(p_\mathrm{T}^\mathrm{miss},\mathbf{p}_{\mathrm{T},\ell})\big)}$.

- Same b-tag as SR ($\geq 2$ b-jets); at least 2 additional jets that are not b-tagged.

- **Data can be plotted here.**


### Z control region (dilepton)

Exactly two OSSF leptons (ee or $\mu\mu$), leading lepton $p_\mathrm{T} > 30$ GeV, $70 < m_{\ell\ell} < 110$ GeV. Data and MC can be compared.

In [ ]:
# Z CR: kinematic preselection (≥1 jet, lead jet pT > 100), Δφ(jet,MET) > 0.5, PF–calo MET < 0.5 GeV, then recoil > 250 GeV; then exactly 2 OSSF leptons, leading lep pT > 30, 70 < m_ll < 110
# Recoil: U = -(pTmiss + sum(lep)); we use |U| for the cut and for the plot
from processor.bbdm_processor import select_tight_electrons, select_tight_muons, recoil_pt, met_pf_calo_consistency_mask

PRESEL_RECOIL_MIN = 250.0
LEAD_JET_PT_MIN = 100.0
Z_CR_MLL_LO, Z_CR_MLL_HI = 70.0, 110.0
LEAD_LEP_PT_CR = 30.0

def z_cr_mask(events):
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    lead_jet_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele = select_tight_electrons(events)
    tight_mu = select_tight_muons(events)
    nele, nmu = ak.count(tight_ele.pt, axis=1), ak.count(tight_mu.pt, axis=1)
    two_ee = (nele == 2) & (nmu == 0) & (ak.sum(tight_ele.charge, axis=1) == 0)
    two_mumu = (nele == 0) & (nmu == 2) & (ak.sum(tight_mu.charge, axis=1) == 0)
    tight_ele_pad = ak.pad_none(tight_ele, 2)
    tight_mu_pad = ak.pad_none(tight_mu, 2)
    pair_ee = tight_ele_pad[:, 0] + tight_ele_pad[:, 1]
    pair_mumu = tight_mu_pad[:, 0] + tight_mu_pad[:, 1]
    sum_lep_px = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.cos(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.cos(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    sum_lep_py = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.sin(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.sin(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    recoil = recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)
    dphi_j = np.abs(good_jets.phi - met_phi)
    dphi_j = ak.where(dphi_j > np.pi, 2 * np.pi - dphi_j, dphi_j)
    min_dphi = ak.min(dphi_j, axis=1)
    dphi_cut = min_dphi > 0.5
    met_pf_calo_ok = met_pf_calo_consistency_mask(events)
    # Order: Δφ, then PF–calo, then recoil |U| (not the other way around)
    presel = (njets >= 1) & (lead_jet_pt > LEAD_JET_PT_MIN) & dphi_cut & met_pf_calo_ok & (recoil > PRESEL_RECOIL_MIN)
    mll_ee = ak.where(two_ee, ak.fill_none(pair_ee.mass, -1.0), ak.full_like(met_pt, -1.0))
    mll_mumu = ak.where(two_mumu, ak.fill_none(pair_mumu.mass, -1.0), ak.full_like(met_pt, -1.0))
    mll = ak.where(two_ee, mll_ee, ak.where(two_mumu, mll_mumu, ak.full_like(met_pt, -1.0)))
    lead_lep_pt = ak.where(two_ee, ak.max(tight_ele.pt, axis=1), ak.where(two_mumu, ak.max(tight_mu.pt, axis=1), ak.full_like(met_pt, 0.0)))
    z_sel = presel & (two_ee | two_mumu) & (lead_lep_pt > LEAD_LEP_PT_CR) & (mll > Z_CR_MLL_LO) & (mll < Z_CR_MLL_HI)
    return z_sel

def z_cr_recoil(events):
    """Recoil in Z CR (for plotting); same lepton sum as in z_cr_mask."""
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele, tight_mu = select_tight_electrons(events), select_tight_muons(events)
    nele, nmu = ak.count(tight_ele.pt, axis=1), ak.count(tight_mu.pt, axis=1)
    two_ee = (nele == 2) & (nmu == 0) & (ak.sum(tight_ele.charge, axis=1) == 0)
    two_mumu = (nele == 0) & (nmu == 2) & (ak.sum(tight_mu.charge, axis=1) == 0)
    tight_ele_pad = ak.pad_none(tight_ele, 2)
    tight_mu_pad = ak.pad_none(tight_mu, 2)
    pair_ee = tight_ele_pad[:, 0] + tight_ele_pad[:, 1]
    pair_mumu = tight_mu_pad[:, 0] + tight_mu_pad[:, 1]
    sum_lep_px = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.cos(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.cos(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    sum_lep_py = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.sin(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.sin(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    return recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)

# Plot recoil in Z CR (data and MC)
fig, ax = plt.subplots()
for name in list(events_by_dataset.keys())[:4]:
    ev = events_by_dataset[name]
    mask = z_cr_mask(ev)
    recoil = z_cr_recoil(ev)[mask]
    if len(ak.ravel(recoil)) == 0:
        continue
    ax.hist(ak.to_numpy(ak.ravel(recoil)), bins=30, range=(250, 600), label=latex_labels.get(name, labels.get(name, name)), alpha=0.6, histtype="step")
ax.set_xlabel("Recoil [GeV]")
ax.set_ylabel("Events")
ax.set_title("Z CR (dilepton): Recoil (data and MC)")
ax.legend()
plt.show()

In [ ]:
# cos(theta*) in Z CR (data and MC)
fig2, ax2 = plt.subplots()
for name in list(events_by_dataset.keys()):
    ev = events_by_dataset[name]
    mask = z_cr_mask(ev)
    good_jets_cr = select_good_jets(ev)[mask]
    has_two = ak.num(good_jets_cr) >= 2
    jets_pad = ak.pad_none(good_jets_cr, 2)
    jet0 = jets_pad[:, 0]
    jet1 = jets_pad[:, 1]
    mask2 = has_two & ~ak.is_none(jet1)
    if ak.sum(mask2) == 0:
        continue
    cts = cos_theta_star(jet0[mask2], jet1[mask2])
    flat = ak.to_numpy(ak.ravel(cts))
    w = norm_factors.get(name)
    weights = np.full_like(flat, w, dtype=float) if (w is not None and not is_data.get(name, False)) else None
    ax2.hist(flat, bins=25, range=(0, 1), weights=weights, label=latex_labels.get(name, labels.get(name, name)), alpha=0.6, histtype="step")
ax2.set_xlabel("$\cos(\theta)*$")
ax2.set_ylabel("Events")
ax2.set_title("Z CR (dilepton): cos $\theta*$ (data and MC)")
ax2.legend()
plt.show()

### Top control region (single-lepton)

Exactly one lepton (e or $\mu$) $p_\mathrm{T} > 30$ GeV; $m_\mathrm{T} < 160$ GeV; $\geq 2$ b-jets; at least 2 additional jets that are not b-tagged. Data and MC can be compared.

In [ ]:
from processor.bbdm_processor import met_pf_calo_consistency_mask

# Top CR: kinematic preselection (≥1 jet, lead jet pT > 100), Δφ(jet,MET) > 0.5, PF–calo MET < 0.5 , then recoil > 250 GeV; then 1 lepton pT > 30, m_T < 160, ≥2 b-jets, ≥2 non-b jets
TOP_CR_MT_MAX = 160.0

def top_cr_mask(events):
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    lead_jet_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele = select_tight_electrons(events)
    tight_mu = select_tight_muons(events)
    one_ele = (ak.count(tight_ele.pt, axis=1) == 1) & (ak.count(tight_mu.pt, axis=1) == 0)
    one_mu = (ak.count(tight_ele.pt, axis=1) == 0) & (ak.count(tight_mu.pt, axis=1) == 1)
    lep_pt = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.pt), ak.where(one_mu, ak.firsts(tight_mu.pt), ak.full_like(met_pt, 0.0))), 0.0)
    lep_phi = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.phi), ak.where(one_mu, ak.firsts(tight_mu.phi), ak.full_like(met_pt, 0.0))), 0.0)
    sum_lep_px = lep_pt * np.cos(lep_phi)
    sum_lep_py = lep_pt * np.sin(lep_phi)
    recoil = recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)
    dphi_j = np.abs(good_jets.phi - met_phi)
    dphi_j = ak.where(dphi_j > np.pi, 2 * np.pi - dphi_j, dphi_j)
    min_dphi = ak.min(dphi_j, axis=1)
    dphi_cut = min_dphi > 0.5
    met_pf_calo_ok = met_pf_calo_consistency_mask(events)
    presel = (njets >= 1) & (lead_jet_pt > LEAD_JET_PT_MIN) & dphi_cut & met_pf_calo_ok & (recoil > PRESEL_RECOIL_MIN)
    dphi = np.abs(ak.to_numpy(met_phi) - ak.to_numpy(lep_phi))
    dphi = np.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    mt = np.sqrt(2.0 * ak.to_numpy(met_pt) * ak.to_numpy(lep_pt) * (1.0 - np.cos(dphi)))
    mt = ak.Array(mt)
    n_non_b = njets - nbjets
    nlep = n_tight_leptons(events)
    return presel & (nlep == 1) & (lep_pt > LEAD_LEP_PT_CR) & (mt < TOP_CR_MT_MAX) & (nbjets >= 2) & (n_non_b >= 2)

def top_cr_recoil(events):
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele, tight_mu = select_tight_electrons(events), select_tight_muons(events)
    one_ele = (ak.count(tight_ele.pt, axis=1) == 1) & (ak.count(tight_mu.pt, axis=1) == 0)
    one_mu = (ak.count(tight_ele.pt, axis=1) == 0) & (ak.count(tight_mu.pt, axis=1) == 1)
    lep_pt = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.pt), ak.where(one_mu, ak.firsts(tight_mu.pt), ak.full_like(met_pt, 0.0))), 0.0)
    lep_phi = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.phi), ak.where(one_mu, ak.firsts(tight_mu.phi), ak.full_like(met_pt, 0.0))), 0.0)
    return recoil_pt(met_pt, met_phi, lep_pt * np.cos(lep_phi), lep_pt * np.sin(lep_phi))

# Plot recoil in Top CR (data and MC)
fig, ax = plt.subplots()
for name in list(events_by_dataset.keys())[:4]:
    ev = events_by_dataset[name]
    mask = top_cr_mask(ev)
    recoil = top_cr_recoil(ev)[mask]
    if len(ak.ravel(recoil)) == 0:
        continue
    ax.hist(ak.to_numpy(ak.ravel(recoil)), bins=30, range=(250, 600), label=latex_labels.get(name, labels.get(name, name)), alpha=0.6, histtype="step")
ax.set_xlabel("Recoil [GeV]")
ax.set_ylabel("Events")
ax.set_title("Top CR (single-lepton): Recoil (data and MC)")
ax.legend()
plt.show()

In [ ]:
# cos(theta*) in Top CR (data and MC)
fig2, ax2 = plt.subplots()
for name in list(events_by_dataset.keys()):
    ev = events_by_dataset[name]
    mask = top_cr_mask(ev)
    good_jets_cr = select_good_jets(ev)[mask]
    has_two = ak.num(good_jets_cr) >= 2
    jets_pad = ak.pad_none(good_jets_cr, 2)
    jet0 = jets_pad[:, 0]
    jet1 = jets_pad[:, 1]
    mask2 = has_two & ~ak.is_none(jet1)
    if ak.sum(mask2) == 0:
        continue
    cts = cos_theta_star(jet0[mask2], jet1[mask2])
    flat = ak.to_numpy(ak.ravel(cts))
    w = norm_factors.get(name)
    weights = np.full_like(flat, w, dtype=float) if (w is not None and not is_data.get(name, False)) else None
    ax2.hist(flat, bins=25, range=(0, 1), weights=weights, label=latex_labels.get(name, labels.get(name, name)), alpha=0.6, histtype="step")
ax2.set_xlabel("$\cos(\theta)*$")
ax2.set_ylabel("Events")
ax2.set_title("Top CR (single-lepton): cos $\\theta*$ (data and MC)")
ax2.legend()
plt.show()

### Discussion
What physics process produces large MET in Z→νν? Why is Z→νν a major background in our signal region?

---
## 8. Using the Coffea Processor

The provided **bbdm_processor.py** does the full selection and fills histograms. You can run it on one or more files to get merged results.

Example (pseudo-code):
```python
from processor.bbdm_processor import bbDMProcessor
from coffea.processor import run_uproot_job
files = {"ttbar": ["file1.root"]}
out = run_uproot_job(files, "events", bbDMProcessor(), executor=iterative_executor)
```
Then inspect `out["recoil"]`, `out["cutflow"]`, etc.

In [ ]:
# Optional: run the processor (uncomment and set paths)
# import sys
# sys.path.insert(0, "processor")
# from bbdm_processor import bbDMProcessor, get_nanoevents
# proc = bbDMProcessor()
# events = get_nanoevents("path/to/file.root")
# result = proc.process(events)
# print("Cutflow:", result["cutflow"])
# result["recoil"].plot()

---
## 9. Comparison Plots

Produce **comparison plots**: e.g. MET distribution **before** any SR cuts (but after ≥1 jet) and **after** full SR selection. This shows how the cut shapes the distribution.

In [ ]:
# Example: MET before and after SR
# met_presel = met[njets >= 1]
# recoil = met[sr_mask]
# fig, ax = plt.subplots(1, 2, figsize=(10, 4))
# ax[0].hist(ak.to_numpy(met_presel), bins=50, range=(0, 500), label="Presel", alpha=0.7)
# ax[1].hist(ak.to_numpy(recoil), bins=50, range=(200, 600), label="SR", alpha=0.7)
# ax[0].set_xlabel("MET [GeV]"); ax[1].set_xlabel("MET [GeV]")
# plt.legend(); plt.tight_layout(); plt.show()

### Exercise 3.4 — Comparison plots
Make a comparison: MET after preselection (≥1 jet) vs MET in signal region. Use two side-by-side or overlaid histograms with legends.

In [ ]:
# Your code: comparison plot

---
## 10. Summary — Session 3

- **Signal region:** Δφ(jet,MET) and PF–calo MET consistency, then **MET > 250 GeV**, ≥2 b-jets (leading two), 0 leptons.
- **Cutflow** and **yield tables** show how each cut reduces events; Z→νν and tt̄ dominate in the SR.
- **Control regions** (e.g. single-lepton) validate background modelling.
- The **bbdm_processor** encapsulates the full selection for reuse.

**End of the long exercise.** You have implemented a simplified bbDM search from data loading to signal region and yields. In a real analysis, next steps would include systematic uncertainties, statistical interpretation, and possibly limit setting.

### Optional: Vary the MET threshold
Re-run your cutflow and MET plot with MET > 150 GeV and MET > 250 GeV. How do the yields and shapes change? Why is the MET cut important for background rejection?